In [ ]:
import pandas as pd
import random
import os
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer


In [ ]:
os.makedirs("data", exist_ok=True)
os.makedirs("model", exist_ok=True)

In [ ]:
def load_jobs():
    df = pd.read_csv("/kaggle/input/your-dataset/data.csv")

    df.columns = df.columns.str.strip().str.lower()

    if "job title" in df.columns:
        title = "job title"
    elif "title" in df.columns:
        title = "title"
    else:
        raise Exception("Job title column not found")

    if "description" in df.columns:
        desc = "description"
    else:
        raise Exception("Description column not found")

    df["job_description"] = df[title].astype(str) + " " + df[desc].astype(str)

    df = df.drop_duplicates(subset=["job_description"])
    df = df.dropna(subset=["job_description"])

    return df[["job_description"]]


In [ ]:
df = load_jobs()

In [ ]:
if "job_description" not in df.columns:
    raise Exception("DataFrame must contain 'job_description' column")

jobs = df["job_description"].tolist()


In [ ]:
data = []

for i in range(len(jobs)):
    resume = jobs[i]

    data.append({
        "resume_text": resume,
        "job_description": jobs[i],
        "label": 1
    })

    random_job = random.choice(jobs)
    if random_job != jobs[i]:
        data.append({
            "resume_text": resume,
            "job_description": random_job,
            "label": 0
        })

labeled_df = pd.DataFrame(data)
labeled_df.to_csv("/kaggle/working/labeled_data.csv", index=False)


In [ ]:
labeled_df["combined"] = labeled_df["resume_text"] + " " + labeled_df["job_description"]

X = labeled_df["combined"]
y = labeled_df["label"]

vectorizer = TfidfVectorizer(max_features=5000)
X_vec = vectorizer.fit_transform(X)

model = LogisticRegression()
model.fit(X_vec, y)

# Save model
joblib.dump(vectorizer, "/kaggle/working/vectorizer.pkl")
joblib.dump(model, "/kaggle/working/classifier.pkl")

print("Model trained successfully")